# TimesFM 3 P1b: sealed policy evaluation

Run after the P1b router state and preferably after Chronos-2 completion have been reviewed. Every routing decision is persisted before inference. TimesFM 3 weights remain restricted to academic, non-commercial use.

Use an **A100** if available; completed origins resume safely.

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/covariate-safe-tsfm')
FEV_CHECKOUT = Path('/content/fev-pinned')
FEV_COMMIT = '38007871dcf6dc6b04aed3a54d9cd86678d48d0b'
TIMESFM_COMMIT = '20191171b74f51bfead932b6b8d0c8f515e70f63'
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
except Exception:
    pass
if not REPO.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/FlyMe2star/covariate-safe-tsfm.git', str(REPO)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
if not FEV_CHECKOUT.exists():
    subprocess.run(
        ['git', 'clone', '--filter=blob:none',
         'https://github.com/autogluon/fev.git', str(FEV_CHECKOUT)],
        check=True,
    )
    subprocess.run(['git', '-C', str(FEV_CHECKOUT), 'checkout', FEV_COMMIT], check=True)
else:
    observed = subprocess.check_output(
        ['git', '-C', str(FEV_CHECKOUT), 'rev-parse', 'HEAD'], text=True
    ).strip()
    assert observed == FEV_COMMIT, observed
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     f'git+https://github.com/autogluon/fev.git@{FEV_COMMIT}',
     ('timesfm[torch] @ git+https://github.com/google-research/'
      f'timesfm.git@{TIMESFM_COMMIT}')],
    check=True,
)
SOURCE_ROOT = str(REPO / 'src')
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)
importlib.invalidate_caches()
importlib.import_module('covsafe')
print('Git commit:', subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True
).strip())
print('Pinned FEV checkout:', FEV_CHECKOUT)
print('HF token available:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
import torch

assert torch.cuda.is_available(), 'Select a GPU runtime, restart, and run all.'
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
PRIVATE_ROOT = Path('/content/drive/MyDrive/covariate-safe-tsfm/private_manifests')
P1B_ROOT = PRIVATE_ROOT / 'p1b'
assert (P1B_ROOT / 'reports/p1b_router_state.json').exists(), (
    'Run notebook 09 and review its router state first.'
)
print('P1b durable root:', P1B_ROOT)

In [ ]:
import json

from covsafe.p1b import EXPECTED_P1B_CONFIG_HASH
from covsafe.timesfm3_p1b import run_timesfm3_p1b

print('Frozen P1b config hash:', EXPECTED_P1B_CONFIG_HASH)
report = run_timesfm3_p1b(REPO, FEV_CHECKOUT, P1B_ROOT, P1B_ROOT)
print(json.dumps(report, indent=2, ensure_ascii=False, default=str))

## Return artifact

Send the completion JSON. If disconnected, reconnect to a GPU and run all cells; verified origins print `RESUME`. Run notebook 12 only after both backbone completion reports have been reviewed.